# 🎬 Motion Video Director - Cloud Render Notebook

**Project:** `myproject`  
**Purpose:** Render motion graphics videos in Google Colab (free tier)  
**Output:** Google Drive `/MyDrive/myproject/renders/`

---

## 📋 Quick Start

1. **Run cells in order** (top to bottom)
2. **Enable GPU:** Runtime → Change runtime type → GPU (T4)
3. **Add Secrets:** Click 🔑 (left sidebar) → Add your API keys
4. **Watch for ✅** - each cell shows success/failure

---

## 🔧 Cell 1: Environment Check & GPU Verify

In [ ]:
# Check GPU availability
import subprocess, sys, torch

print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU - will use CPU (slower)")

# Check ffmpeg
result = subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True)
if result.returncode == 0:
    print(f"ffmpeg: {result.stdout.split('\n')[0]}")
else:
    print("❌ ffmpeg not found")


---

## 🔐 Cell 2: Load Secrets from Colab Userdata

**Before running:** Click the 🔑 key icon (left sidebar) → Add these secrets:
- `PIXABAY_API_KEY`
- `FREESOUND_API_KEY`
- `FREESOUND_CLIENT_ID`
- `PEXELS_API_KEY`
- `UNSPLASH_ACCESS_KEY`
- `ELEVENLABS_API_KEY`
- `NIM_API_KEY` (optional, for planning only)

*Secrets are never printed or logged.*

In [ ]:
# Load secrets from Colab userdata
import os
from google.colab import userdata

SECRET_KEYS = [
    "PIXABAY_API_KEY",
    "FREESOUND_API_KEY",
    "FREESOUND_CLIENT_ID",
    "PEXELS_API_KEY",
    "UNSPLASH_ACCESS_KEY",
    "ELEVENLABS_API_KEY",
    "NIM_API_KEY",
    "NIM_BASE_URL"
]

loaded = {}
for key in SECRET_KEYS:
    try:
        value = userdata.get(key)
        if value:
            os.environ[key] = value
            loaded[key] = value
    except Exception:
        pass

print("🔐 Secrets loaded:")
for key in SECRET_KEYS:
    status = "✅" if key in loaded else "⬜"
    print(f"  {status} {key}")

missing = [k for k in SECRET_KEYS if k not in loaded]
if missing:
    print(f"\n⚠️  Missing: {', '.join(missing)} (add in 🔑 Secrets panel)")

---

## ☁️ Cell 3: Mount Google Drive

This creates persistent storage at `/content/drive/MyDrive/myproject/`.
All final renders and previews will be saved here automatically.

In [ ]:
# Mount Google Drive
from google.colab import drive

DRIVE_FOLDER = "MyDrive/myproject"
drive.mount('/content/drive', force_remount=True)

drive_path = Path(f"/content/drive/{DRIVE_FOLDER}")
drive_path.mkdir(parents=True, exist_ok=True)

# Create symlink for easy access
project_link = Path("/content/project")
if project_link.exists() or project_link.is_symlink():
    project_link.unlink()
project_link.symlink_to(drive_path)

print(f"☁️  Drive mounted: {drive_path}")
print(f"🔗 Symlink: /content/project -> {drive_path}")
print(f"📁 Contents: {list(drive_path.iterdir()) if drive_path.exists() else 'empty'}")

---

## 📥 Cell 4: Clone/Sync GitHub Repository

**Edit the REPO_URL below** with your private GitHub repository URL.
Format: `https://github.com/USERNAME/REPO.git`

*For private repos, you'll need to authenticate. See troubleshooting below.*

In [ ]:
# Clone or pull repository
import subprocess
from pathlib import Path

# >>> EDIT THIS <<<
REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"

project_dir = Path("/content/project")

if (project_dir / ".git").exists():
    print("📦 Repository exists, pulling latest...")
    result = subprocess.run(["git", "-C", str(project_dir), "pull"], capture_output=True, text=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
else:
    print(f"📦 Cloning from {REPO_URL}...")
    result = subprocess.run(["git", "clone", REPO_URL, str(project_dir)], capture_output=True, text=True)
    if result.returncode == 0:
        print("✅ Clone successful")
    else:
        print("❌ Clone failed:")
        print(result.stderr)
        print("\n🔧 For private repos, run this first:")
        print("   !git config --global credential.helper store")
        print("   Then enter your GitHub PAT when prompted")

---

## ⚙️ Cell 5: Install Dependencies & Setup Environment

This installs all Python packages, Playwright Chromium, and downloads ML models.
**Takes 2-5 minutes on first run.**

In [ ]:
# Run setup script
import subprocess, sys

setup_script = Path("/content/project/scripts/colab_setup.py")
if setup_script.exists():
    result = subprocess.run([
        sys.executable, str(setup_script),
        "--repo", REPO_URL,
        "--drive-folder", "MyDrive/myproject"
    ], capture_output=False, text=True)
else:
    print("❌ Setup script not found. Check repository structure.")

---

## 📦 Cell 6: Download Assets from Manifest

Downloads all fonts, images, SFX, and music listed in `manifest.json`.
Skips files that already exist.

In [ ]:
# Download all assets
import subprocess, sys

fetch_script = Path("/content/project/scripts/fetch_assets.py")
result = subprocess.run([
    sys.executable, str(fetch_script),
    "--category", "all"
], capture_output=False, text=True)

---

## 👀 Cell 7: Preview Render (Fast, Low-Res)

**Always run this before final render!**
- 8 fps, 25% resolution (270x480)
- Generates contact sheet (JPG grid) + preview MP4
- Takes ~15-30 seconds
- Check framing, timing, text legibility

In [ ]:
# Render preview
import subprocess, sys
from pathlib import Path

render_script = Path("/content/project/scripts/colab_render.py")
result = subprocess.run([
    sys.executable, str(render_script),
    "--mode", "preview",
    "--from", "0",
    "--to", "40",
    "--contact-sheet"
], capture_output=False, text=True)

# Display contact sheet
from IPython.display import Image, display
sheet_path = Path("/content/project/renders/previews/sheet_0-40.jpg")
if sheet_path.exists():
    print(f"📋 Contact sheet: {sheet_path}")
    display(Image(filename=str(sheet_path)))
else:
    print("❌ Contact sheet not found")

# Show preview video
preview_path = Path("/content/project/renders/previews/preview_0-40.mp4")
if preview_path.exists():
    print(f"🎬 Preview video: {preview_path}")
    display(Image(filename=str(preview_path)))
    # For actual video playback, download from Drive
else:
    print("❌ Preview video not found")

---

## 🎯 Cell 8: Final Render (Full Quality)

**Run only after preview approval!**
- 30 fps, 100% resolution (1080x1920)
- H.264 CRF 18, slow preset
- Loudness normalized to -14 LUFS
- Auto-chunks if > 2 minutes
- Saves to Google Drive automatically

⏱️ **Estimated time:** 2-10 minutes depending on complexity

In [ ]:
# Render final video
import subprocess, sys
from pathlib import Path

render_script = Path("/content/project/scripts/colab_render.py")
result = subprocess.run([
    sys.executable, str(render_script),
    "--mode", "final",
    "--from", "0",
    "--to", "40",
    "--chunked",
    "--max-chunk-duration", "120"
], capture_output=False, text=True)

# Find and display output
final_dir = Path("/content/drive/MyDrive/myproject/renders/final")
final_files = sorted(final_dir.glob("myproject_v*.mp4"))
if final_files:
    latest = final_files[-1]
    print(f"\n✅ Final render complete!")
    print(f"📁 Output: {latest}")
    print(f"📏 Size: {latest.stat().st_size / 1e6:.1f} MB")
    
    # Verify with ffprobe
    import subprocess
    probe = subprocess.run([
        "ffprobe", "-v", "error",
        "-show_entries", "format=duration,bit_rate",
        "-of", "default=noprint_wrappers=1:nokey=1",
        str(latest)
    ], capture_output=True, text=True)
    if probe.returncode == 0:
        lines = probe.stdout.strip().split('\n')
        print(f"⏱️  Duration: {lines[0]}s")
        print(f"📊 Bitrate: {int(lines[1])/1000:.0f} kbps")
else:
    print("❌ No final output found")

---

## 📤 Cell 9: Download Final Video

Click the file below to download, or get it from Google Drive:
`/MyDrive/myproject/renders/final/`

In [ ]:
# List all renders for download
from pathlib import Path
from IPython.display import FileLink, display

final_dir = Path("/content/drive/MyDrive/myproject/renders/final")
preview_dir = Path("/content/project/renders/previews")

print("📁 Final Renders (Google Drive):")
for f in sorted(final_dir.glob("*.mp4")):
    size_mb = f.stat().st_size / 1e6
    display(FileLink(str(f), result_html_prefix=f"🎬 {f.name} ({size_mb:.1f} MB): "))

print("\n📁 Previews (Colab temp):")
for f in sorted(preview_dir.glob("*.mp4")):
    size_mb = f.stat().st_size / 1e6
    display(FileLink(str(f), result_html_prefix=f"👀 {f.name} ({size_mb:.1f} MB): "))

print("\n📋 Contact Sheets:")
for f in sorted(preview_dir.glob("sheet_*.jpg")):
    display(FileLink(str(f), result_html_prefix=f"📋 {f.name}: "))

---

## 🛠️ Troubleshooting & Tips

### Private GitHub Repo Authentication
```bash
# In a cell, run:
!git config --global credential.helper store
# Then clone - enter username and Personal Access Token (not password)
```

### Out of Memory (OOM) Errors
- Reduce `--scale` in preview (0.15 instead of 0.25)
- Enable `--chunked` with smaller `--max-chunk-duration` (60s)
- Close other Colab tabs

### Runtime Disconnects
- Colab free tier: 12h max, 90min idle timeout
- Use chunked rendering - can resume from last chunk
- Save checkpoints: `drive.flush_and_unmount()` then re-mount

### Missing Fonts
- Run `fetch_assets.py --category fonts` first
- Fonts must be in `assets/fonts/` with .woff2/.ttf extension
- Check `manifest.json` for correct paths

### Audio Sync Issues
- Ensure VO file is at `assets/vo/vo_final.mp3`
- Check `beats.json` matches VO timing
- Run `python scripts/analyze_ref.py assets/vo/vo_final.mp3` to verify

### GPU Not Available
- Runtime → Change runtime type → GPU (T4)\n- Restart runtime after changing\n- CPU rendering works but 5-10x slower\n
---

## 📚 Resources
- **Full Guide:** `COLAB_RENDER.md` in repository
- **Skill Docs:** Motion Video Director skill (§2 workflow)
- **Project Files:** `/content/project/` (synced with Drive)
- **Logs:** Check cell outputs above for errors

---

*Happy rendering! 🎬*